# Earnings RAG demo

A hands-on walkthrough of `earnings_rag.py` on IBM's Q1 and Q2 2026 earnings calls.
Each step calls the same functions the CLI uses and prints what they produce, so you can see
the data at every stage: turns → chunks → vectors → database rows → retrieved excerpts → cited answer.

| Step | What you'll see |
|---|---|
| 1 | Raw transcript from Alpha Vantage |
| 2 | Speaker-aware chunks and their ids |
| 3 | Embeddings and cosine similarity |
| 4 | Storing chunks in Milvus (default) |
| 5 | Upsert is idempotent |
| 6 | Hybrid search with metadata filters |
| 7 | Effect of the cross-encoder rerank |
| 8 | Grounded answer with citations and audit |
| 9 | Same query across Milvus, Qdrant and in-memory |
| 10 | The equivalent CLI commands |

**Before you start**
- Put this notebook in the same folder as `earnings_rag.py`.
- Free Alpha Vantage key: https://www.alphavantage.co/support/#api-key
- An Anthropic key is only needed for step 8; every other step runs without it.
- First run downloads two small models from Hugging Face (about 130 MB in total).

In [ ]:
# Run once. Remove the leading "#" to install.
%pip install requests rank_bm25 numpy sentence-transformers anthropic pymilvus milvus-lite qdrant-client

In [1]:
import sys; print(sys.executable, sys.version)

D:\conda_envs\finder\python.exe 3.11.16 | packaged by Anaconda, Inc. | (main, Aug 27 2026, 14:36:16) [MSC v.1942 64 bit (AMD64)]


## 0. Setup

In [2]:
import os, time
# Set keys here or in your shell before starting Jupyter.
# os.environ["ALPHAVANTAGE_API_KEY"] = "your-key"
# os.environ["ANTHROPIC_API_KEY"]    = "your-key"     # only for step 8

import earnings_rag as rag
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("ALPHAVANTAGE_API_KEY")

print (api_key)
DB = "milvus"                              # try "qdrant" or "memory" too
PAIRS = [("IBM", "2026Q2"), ("IBM", "2026Q1")]

print("Alpha Vantage key set:", bool(os.getenv("ALPHAVANTAGE_API_KEY")))
print("Anthropic key set:    ", bool(os.getenv("ANTHROPIC_API_KEY")))
print("Vector store:         ", DB)

H7WVPJU5CYCM5W30
Alpha Vantage key set: True
Anthropic key set:     False
Vector store:          milvus


D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## 1. Fetch a transcript
`fetch_transcript` calls Alpha Vantage once and caches the JSON in `./transcripts/`.
Running the cell again reads the cache, so it costs no API quota.

In [3]:
turns = rag.fetch_transcript("IBM", "2026Q2")
print(f"{len(turns)} speaker turns\n")
for t in turns[:6]:
    print(f"{t['speaker']:<22} {t['title'][:45]:<45} sentiment={t['sentiment']}  words={len(t['content'].split())}")

23 speaker turns

Operator               Operator                                      sentiment=0.0  words=43
Olympia McNerney       Global Head of Investor Relations             sentiment=0.0  words=176
Arvind Krishna         Chairman, President and Chief Executive Offic sentiment=0.7  words=1392
James Kavanaugh        Senior Vice President and Chief Financial Off sentiment=0.6  words=1767
Olympia McNerney       Global Head of Investor Relations             sentiment=0.0  words=45
Amit Daryanani         Analyst                                       sentiment=0.0  words=151


You should see the Operator, Investor Relations, then the CEO and CFO. The CEO and CFO turns are long, so they will be split into several chunks.

## 2. Chunk it
Rules in `chunk_transcript`: one speaker per chunk, 300-word windows with 50 words of overlap,
Operator turns dropped, and `section` switches from `prepared` to `qa` at the first Analyst.

In [4]:
from collections import Counter
chunks = rag.chunk_transcript(turns, "IBM", "2026Q2")

print(len(chunks), "chunks by section:", dict(Counter(c.section for c in chunks)))
print("chunks per speaker:", dict(Counter(c.speaker for c in chunks)))
print("\nFirst Q&A chunk:")
first_qa = next(c for c in chunks if c.section == "qa")
print(first_qa)
print("\nText that gets embedded and indexed:")
print(rag.chunk_text(first_qa)[:300], "...")

45 chunks by section: {'prepared': 15, 'qa': 30}
chunks per speaker: {'Olympia McNerney': 3, 'Arvind Krishna': 16, 'James Kavanaugh': 20, 'Amit Daryanani': 1, 'Brent Thill': 1, 'Benjamin Reitzes': 1, 'Fatima Boolani': 1, 'Erik Woodring': 1, 'Matthew Swanson': 1}

First Q&A chunk:
Chunk(text="I guess, Arvind, your commentary strongly suggests that the headwinds we saw in June quarter were a result of customers just reallocating spend towards AI infrastructure rather than reducing overall IT budgets. And I think the biggest debate everyone has since the pre-announcement has been, is this demand deferred or destroyed for IBM? So I'd love to understand what gives you the confidence that this is more a timing issue versus a structural shift in enterprise spending? And what milestones should we all be watching for over the next couple of quarters to gauge if demand truly starts to normalize for you folks? That would be really helpful. And Jim, I have to acknowledge the very impressive free c

Look at the ids: `IBM-2026Q2-t3-0` means ticker, quarter, turn 3, window 0. They are built from the data, so the same transcript always produces the same ids. Step 5 relies on that.

## 3. Embed
`Embedder` wraps `BAAI/bge-small-en-v1.5`: 384 numbers per chunk, scaled to length 1 so a dot product equals cosine similarity.

In [5]:
import numpy as np
emb = rag.Embedder()
vecs = emb.embed([rag.chunk_text(c) for c in chunks])
print("matrix shape:", vecs.shape, " row norms:", np.round(np.linalg.norm(vecs[:3], axis=1), 3))

q = emb.embed(["Is the demand deferred or lost?"])[0]
sims = vecs @ q                                   # cosine similarity with every chunk
for i in np.argsort(-sims)[:3]:
    print(f"{sims[i]:.3f}  {chunks[i].id:<18} {chunks[i].speaker}: {chunks[i].text[:80]}...")

D:\conda_envs\finder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3755.85it/s]
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].
'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json
Retrying in 4s [Retry 3/5].
'[WinError 10054] An existing 

matrix shape: (45, 384)  row norms: [1. 1. 1.]
0.754  IBM-2026Q2-t8-1    Arvind Krishna: clients in terms of what they're purchasing and putting in, then I come to the c...
0.734  IBM-2026Q2-t8-0    Arvind Krishna: Thanks, Jim. So Amit, let me address the first part of your question. I have to ...
0.727  IBM-2026Q2-t5-0    Amit Daryanani: I guess, Arvind, your commentary strongly suggests that the headwinds we saw in ...


This is dense-only search. Steps 6 and 7 add BM25, RRF fusion and reranking on top.

## 4. Store in the vector database
`ingest` = fetch → chunk → embed → `store.create()` → `store.upsert()`.
`recreate=True` drops the collection first so the demo starts clean.

In [6]:
store = rag.make_store(DB) if DB != "memory" else None
if store:
    n = rag.ingest(store, emb, PAIRS, recreate=True)
    print(f"\n[{DB}] upserted {n} chunks, collection holds {store.count()}")
else:
    print("memory mode keeps nothing on disk; step 6 builds the index in RAM")

  IBM 2026Q2: 45 chunks
  IBM 2026Q1: 47 chunks

[milvus] upserted 92 chunks, collection holds 92


## 5. Upsert is idempotent
Ingest Q2 again. Every chunk id already exists, so each write overwrites a row instead of adding one.
The count should not change.

In [7]:
if store:
    before = store.count()
    rag.ingest(store, emb, [("IBM", "2026Q2")])
    print(f"before={before}  after={store.count()}  (same number means no duplicates)")

  IBM 2026Q2: 45 chunks
before=92  after=92  (same number means no duplicates)


## 6. Hybrid search with filters
`StoreRetriever` embeds the question and calls `store.search()`. Inside Milvus, a dense route and a BM25 route
run with the same filter, and `RRFRanker(60)` merges them. Rerank is off here so you see the database's own ranking.

In [8]:
retriever = (rag.build_retriever("memory", PAIRS, use_rerank=False) if DB == "memory"
             else rag.StoreRetriever(store, emb, use_rerank=False))

def show(hits):
    for c, s in hits:
        print(f"  {s:7.4f}  {c.id:<18} [{c.section:<8}] {c.speaker}: {c.text[:90]}...")

queries = [
    ("What is the full-year revenue guidance?",           {"symbol": "IBM", "quarter": "2026Q2"}),
    ("Is the demand deferred or destroyed?",              {"symbol": "IBM", "quarter": "2026Q2", "section": "qa"}),
    ("ARR $24.6 billion",                                 {"symbol": "IBM"}),                       # exact figure: BM25 helps
    ("How did Red Hat grow?",                             {"symbol": "IBM", "quarter": ["2026Q1", "2026Q2"]}),  # list = OR
]
for qtext, flt in queries:
    print(f"\nQ: {qtext}\n   filters={flt}")
    show(retriever.search(qtext, k=3, filters=flt))


Q: What is the full-year revenue guidance?
   filters={'symbol': 'IBM', 'quarter': '2026Q2'}
   0.0323  IBM-2026Q2-t3-6    [prepared] James Kavanaugh: drive more efficient software development, optimizing our supply chain and enhancing servi...
   0.0323  IBM-2026Q2-t10-0   [qa      ] Brent Thill: Jim, I just want to go through the guidance for the rest of the year. So your prior guide ...
   0.0312  IBM-2026Q2-t3-4    [prepared] James Kavanaugh: headwind mitigated somewhat by strong Distributed Infrastructure margins. Through the firs...

Q: Is the demand deferred or destroyed?
   filters={'symbol': 'IBM', 'quarter': '2026Q2', 'section': 'qa'}
   0.0328  IBM-2026Q2-t8-1    [qa      ] Arvind Krishna: clients in terms of what they're purchasing and putting in, then I come to the conclusion ...
   0.0320  IBM-2026Q2-t5-0    [qa      ] Amit Daryanani: I guess, Arvind, your commentary strongly suggests that the headwinds we saw in June quart...
   0.0320  IBM-2026Q2-t8-0    [qa      ] Arv

What to check:
- The guidance query should surface the CFO's prepared remarks (4% to 5% revenue growth, about $1 billion more free cash flow).
- The `section: qa` query only returns Q&A turns, including the analyst question itself.
- The `ARR $24.6 billion` query finds the exact figure; that is the BM25 route doing its job.
- The Red Hat query can return chunks from both quarters because the filter value is a list.

## 7. Rerank on vs off
The cross-encoder reads the question and each passage together. It re-scores the database's top 20 and keeps the best k.

In [9]:
qtext = "Why did software revenue miss expectations?"
flt = {"symbol": "IBM", "quarter": "2026Q2"}
reranked = (rag.build_retriever("memory", PAIRS, use_rerank=True) if DB == "memory"
            else rag.StoreRetriever(store, emb, use_rerank=True))

print("Without rerank (RRF order):");   show(retriever.search(qtext, k=4, filters=flt))
print("\nWith rerank (cross-encoder):"); show(reranked.search(qtext, k=4, filters=flt))

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3254.02it/s]


Without rerank (RRF order):
   0.0318  IBM-2026Q2-t17-0   [qa      ] Erik Woodring: Sorry about that, guys. So Arvind, I want to touch on the data segment within software. An...
   0.0318  IBM-2026Q2-t18-0   [qa      ] Arvind Krishna: Erik, thanks for the question. First, let's understand. As we said, in aggregate, about $4...
   0.0318  IBM-2026Q2-t14-0   [qa      ] James Kavanaugh: Yes. The questions kind of go together, right, because we run this as a mainframe stack pl...
   0.0318  IBM-2026Q2-t3-1    [prepared] James Kavanaugh: the quarter. Let me take a step back and discuss the composition of our software revenue. ...

With rerank (cross-encoder):
   2.6224  IBM-2026Q2-t3-5    [prepared] James Kavanaugh: revenue range appropriately reflects the current environment and is our base case. Given t...
   1.3006  IBM-2026Q2-t3-6    [prepared] James Kavanaugh: drive more efficient software development, optimizing our supply chain and enhancing servi...
   0.8648  IBM-2026Q2-t2-0    [pr

Scores change scale after reranking (cross-encoder logits instead of RRF sums). Compare the order, not the numbers.

## 8. Grounded answer with citations
`answer()` sends the top excerpts to Claude with rules: use only the excerpts, cite every claim by id, quote numbers exactly.
It then checks the citations: `hallucinated_ids` lists ids the model cited but never received, and should always be empty.

In [ ]:
if os.getenv("ANTHROPIC_API_KEY"):
    out = rag.answer(reranked, "What caused the Q2 software shortfall, and is the demand deferred or lost?",
                     k=8, filters={"symbol": "IBM", "quarter": "2026Q2"})
    print(out["answer"])
    print("\ncited sources:   ", [s["id"] for s in out["sources"]])
    print("uncited_ids:     ", out["uncited_ids"])
    print("hallucinated_ids:", out["hallucinated_ids"], "(should be empty)")
else:
    print("Set ANTHROPIC_API_KEY to run this step. Showing the prompt the model would receive instead:\n")
    hits = reranked.search("What caused the Q2 software shortfall?", k=2, filters={"symbol": "IBM", "quarter": "2026Q2"})
    print(rag.format_context(hits)[:1200], "...")

## 9. Same query across stores
Each database ranks a little differently. Compare which chunk ids come back and how long the search takes.
This is the start of the recall@k / latency check described in the "which database" notes.

In [10]:
qtext, flt = "Is the demand deferred or destroyed?", {"symbol": "IBM", "quarter": "2026Q2", "section": "qa"}
for db in ["milvus", "qdrant", "memory"]:
    try:
        r = rag.build_retriever(db, PAIRS, embedder=emb, use_rerank=False)   # ingests for db stores
        t0 = time.perf_counter(); hits = r.search(qtext, k=3, filters=flt); ms = (time.perf_counter() - t0) * 1000
        print(f"{db:<7} {ms:6.1f} ms  ", [c.id for c, _ in hits])
    except ImportError as e:
        print(f"{db:<7} skipped: {e.name} not installed")

  IBM 2026Q2: 45 chunks
  IBM 2026Q1: 47 chunks
milvus    46.5 ms   ['IBM-2026Q2-t8-1', 'IBM-2026Q2-t5-0', 'IBM-2026Q2-t8-0']


D:\Fin_Rag\notebooks\earnings_rag.py:364: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  self.client.create_payload_index(self.name, f, m.PayloadSchemaType.KEYWORD)


  IBM 2026Q2: 45 chunks
  IBM 2026Q1: 47 chunks
qdrant    47.2 ms   ['IBM-2026Q2-t8-0', 'IBM-2026Q2-t8-1', 'IBM-2026Q2-t5-0']


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2678.05it/s]


memory    24.5 ms   ['IBM-2026Q2-t8-1', 'IBM-2026Q2-t5-0', 'IBM-2026Q2-t8-0']


In [11]:
store.client.close()          # the Milvus connection from step 4
print("closed")

closed


Timings on a few dozen chunks are dominated by fixed overhead. Differences only become meaningful at tens of thousands of chunks and above.

## 10. The same thing from the command line
Everything above is available through the CLI. Run these in a terminal from the same folder.
```bash
# store once (Milvus is the default; add --db qdrant to switch)
python earnings_rag.py ingest IBM:2026Q2 IBM:2026Q1

# retrieval only, no Anthropic key needed
python earnings_rag.py ask "Is the demand deferred or destroyed?" --symbol IBM --quarter 2026Q2 --section qa --no-llm

# full answer with citations
python earnings_rag.py ask "What is the full-year revenue guidance?" --symbol IBM --quarter 2026Q2

# start over with a clean collection
python earnings_rag.py ingest IBM:2026Q2 --recreate
```

**Try next**
- Add another company: `PAIRS.append(("MSFT", "2026Q2"))`, re-run step 4, then filter by `symbol`.
- Edit the `max_words` default in `chunk_transcript` (in earnings_rag.py) to 150, restart the kernel, and re-ingest without `--recreate`. Count the rows: the old extra windows stay behind as orphans, which is the upsert caveat.
- Write 10 questions with the chunk ids you expect, and measure how often each store returns them in the top 3.